<div dir="rtl">
<h1>دو پاسخ، یک ترجیح قابل آزمایش</h1>
<p>درس 75 از 76 · ترجیح دو پاسخ چه سیگنالی به مدل می‌دهد؟ · <code dir="ltr">66-preference</code></p>
<p><a target="_self" href="http://127.0.0.1:8000/part-10/chapter-02/66-preference.html">📖 بازگشت به همین درس</a></p>
<p>اثر ترجیح را در یک مسئلهٔ دوپاسخی کوچک ببینید؛ آن را با سنجش حقیقت اشتباه نگیرید.</p><p>پیش‌نیاز: Log Probability، Softmax، Gradient و مدل مرجع ثابت.</p>
<p>این دفتر نیمهٔ عملی درس است. مثال‌ها آمادهٔ اجرا هستند؛ دو Cell با برچسب TODO را خودتان کامل کنید. پیام INCOMPLETE یعنی هنوز چیزی ننوشته‌اید، نه اینکه پاسخ درست است. جواب مرجع در این دفتر پنهان نشده است.</p>
<p>از بالا به پایین اجرا کنید. پس از تغییر هر تابع، Cell آن و سپس Cell آزمون را دوباره اجرا کنید. برای بررسی نهایی، از منوی <code>Kernel → Restart Kernel and Run All Cells</code> استفاده کنید.</p>
</div>

In [ ]:
from pathlib import Path
import os
import sys

project_root = next((p for p in (Path.cwd(), *Path.cwd().parents)
                     if (p / "mini_gpt").is_dir() and (p / "book_src").is_dir()), None)
if project_root is None:
    raise RuntimeError("Extract the complete learning project; open this notebook inside it.")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
print("Python:", sys.executable)
print("Project:", project_root)

<div dir="rtl">
<h2>قبل از اجرا، پیش‌بینی کنید</h2>
<p>اگر دادهٔ ترجیح پاسخ غلط را برنده اعلام کند، آیا تابع هدف خودش این اشتباه را کشف می‌کند؟</p>
</div>

<div dir="rtl"><p>پیش‌بینی من: …</p></div>

In [ ]:
import math
import torch
from torch.nn import functional as F
torch.set_num_threads(1)
responses = ['پاسخ درست و کوتاه','پاسخ روان اما غلط']
reference_logits = torch.tensor([0.0,0.0])
policy_logits = torch.tensor([0.0,0.0],requires_grad=True)
print(list(zip(responses,policy_logits.softmax(-1).detach().tolist())))
print('Each outcome here is a whole answer, not one token from MiniGPT.')

<div dir="rtl">
<h2>این بار شما کد بنویسید</h2>
<p>preference_loss(policy_logits,reference_logits,chosen,beta) را برای دقیقاً دو پاسخ بنویسید. chosen اندیس پاسخ ترجیح‌داده‌شده است و دیگری 1-chosen. log_softmax هر دو توزیع را بگیرید. margin اختلاف log-prob برنده و بازنده در Policy، منهای همین اختلاف در مرجع ثابت است. Loss برابر softplus(-beta*margin) است؛ softplus(z)=log(1+exp(z)) و نسخهٔ PyTorch پایدارتر است. این نمونهٔ محدودِ هدف DPO است، نه فرایند کامل آموزش مدل زبان.</p>
</div>

In [ ]:
def preference_loss(policy_logits, reference_logits, chosen, beta):
    # TODO: مقایسهٔ نسبی با مرجع ثابت
    return None

In [ ]:
def test_exercise():
    result = preference_loss(policy_logits,reference_logits,0,0.5)
    if result is None:
        return False
    assert abs(result.item()-math.log(2))<1e-6
    favored = torch.tensor([2.0,0.0])
    assert preference_loss(favored,reference_logits,0,0.5)<result
    assert preference_loss(favored,reference_logits,1,0.5)>result
    torch.testing.assert_close(preference_loss(favored,favored,0,0.5),torch.tensor(math.log(2)))
    candidate = torch.tensor([0.0,0.0],requires_grad=True)
    ref = torch.tensor([0.0,0.0],requires_grad=True)
    preference_loss(candidate,ref,0,0.5).backward()
    assert candidate.grad[0]<0 and candidate.grad[1]>0
    assert ref.grad is None
    return True
exercise_complete = test_exercise()
print('PASS' if exercise_complete else 'INCOMPLETE: preference_loss')

<div dir="rtl">
<h2>فقط یک عامل را تغییر دهید</h2>
<p>فقط برچسب ترجیح را عوض کنید؛ دو پاسخ و Policy آغازین ثابت‌اند. فرمول زیر صرفاً یک گام عددی مستقل را نشان می‌دهد، نه آزمون درستی پاسخ.</p>
</div>

In [ ]:
for chosen in (0,1):
    scores = torch.tensor([0.0,0.0],requires_grad=True)
    logp = scores.log_softmax(-1)
    margin = logp[chosen]-logp[1-chosen]
    F.softplus(-0.5*margin).backward()
    with torch.no_grad():
        scores -= 0.2*scores.grad
    print('preferred:',chosen,'new probabilities:',scores.softmax(-1).detach().tolist())

<div dir="rtl">
<h2>خرابی را پیدا کنید</h2>
<p>اگر ترتیب برنده و بازنده را برعکس کنید، کم‌کردن Loss پاسخ نامطلوب را تقویت می‌کند. preference_margin(policy_logps,reference_logps,chosen,rejected) اختلاف درست نسبت به مرجع را برگرداند؛ ورودی‌ها از قبل log-prob هستند.</p>
</div>

In [ ]:
scores = torch.tensor([0.0,0.0],requires_grad=True)
logp = scores.log_softmax(-1)
wrong_margin = logp[1]-logp[0]  # The data actually prefers answer 0.
F.softplus(-wrong_margin).backward()
with torch.no_grad():
    scores -= 0.1*scores.grad
print('wrong-sign update:',scores.softmax(-1).tolist())

<div dir="rtl">
<h2>اصلاح را خودتان بنویسید</h2>
<p>علت را توضیح دهید، سپس تابع زیر را کامل کنید. خطای عمدی بالا یک نمونهٔ آموزشی است؛ آزمون پایین باید اصلاح شما را بسنجد.</p>
</div>

In [ ]:
def preference_margin(policy_logps, reference_logps, chosen, rejected):
    # TODO: برنده منهای بازنده، در هر دو مدل
    return None

In [ ]:
def test_repair():
    p = torch.tensor([0.8,0.2]).log()
    r = torch.tensor([0.5,0.5]).log()
    result = preference_margin(p,r,0,1)
    if result is None:
        return False
    torch.testing.assert_close(result,torch.tensor(math.log(4)))
    torch.testing.assert_close(preference_margin(p,r,1,0),torch.tensor(-math.log(4)))
    torch.testing.assert_close(preference_margin(r,r,0,1),torch.tensor(0.0))
    return True
repair_complete = test_repair()
print('PASS' if repair_complete else 'INCOMPLETE: preference_margin')

<div dir="rtl">
<h2>در Mini-GPT کجا به کار می‌آید؟</h2>
<p>این آزمایش دوپاسخی از صورت هدف در <a href='https://arxiv.org/abs/2305.18290'>مقالهٔ DPO</a> استفاده می‌کند. در مدل زبان، log-prob پاسخ از Tokenهای پاسخ به دست می‌آید. MiniGPT فعلی این فرایند، مدل پاداش یا حلقهٔ RLHF ندارد؛ DPO و RLHF دو ایستگاه اجباری پشت‌سرهم نیستند.</p>
</div>

<div dir="rtl">
<h2>با زبان خودتان توضیح دهید</h2>
<p>اگر ترجیح‌ها معیار بدی داشته باشند، کاهش این Loss چه چیزی را بهتر می‌کند و چه چیزی را تضمین نمی‌کند؟</p>
</div>
<div dir="rtl"><p>پیش‌بینی و مشاهدهٔ من: …</p><p>علت خرابی و اصلاح من: …</p></div>

<div dir="rtl"><p><a target="_self" href="http://127.0.0.1:8000/part-10/chapter-02/66-preference.html">بازگشت به درس و ادامهٔ مسیر</a> · <a target="_self" href="http://127.0.0.1:8000/answers/66-preference.html#lab-solution">فقط پس از تلاش: راه‌حل مرجع آزمایشگاه</a></p></div>